## Feature Engineering Pipeline for New Data
This notebook applies the same cleaning, winsorization, logs, flags, and engineered features used in `eda.ipynb`, with fixed thresholds learned from your training data.


In [ ]:
import numpy as np
import pandas as pd


In [ ]:
# Fixed winsorization thresholds from EDA training run
CAPS = {
    'adr': 285.0,
    'lead_time': 347.0,
    'stays_in_weekend_nights': 4.0,
    'stays_in_week_nights': 10.0,
    'booking_changes': 4.0,
    'days_in_waiting_list': 58.0,
    'previous_cancellations': 2.0,
    'previous_bookings_not_canceled': 9.0,
}

# Business-rule caps used in EDA
LIMITS = {
    'adults': 6,
    'children': 3,
    'babies': 2,
}

MONTH_TO_NUM = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8,
    'September': 9, 'October': 10, 'November': 11, 'December': 12
}


In [ ]:
def preprocess_new_data(df_raw, country_lookup=None, drop_duplicates=True):
    # Apply the same preprocessing logic as EDA to new/raw data.
    df_clean = df_raw.copy()

    # ---- Base cleaning (same as EDA) ----
    df_clean['children'] = df_clean['children'].fillna(0).astype(int)
    df_clean['country'] = df_clean['country'].fillna('Unknown')

    df_clean['agent'] = (
        df_clean['agent']
        .replace(['NULL', 'null', 'Null'], np.nan)
        .fillna('Direct')
        .astype(str)
    )
    df_clean['company'] = (
        df_clean['company']
        .replace(['NULL', 'null', 'Null'], np.nan)
        .fillna('None')
        .astype(str)
    )

    # Remove phantom bookings
    phantom_mask = (
        (df_clean['adults'] == 0)
        & (df_clean['children'] == 0)
        & (df_clean['babies'] == 0)
    )
    df_clean = df_clean.loc[~phantom_mask].copy()

    # Remove negative ADR
    df_clean = df_clean.loc[df_clean['adr'] >= 0].copy()

    # ---- Winsorization + flags/logs (same structure as EDA) ----
    # ADR
    df_clean['adr_was_capped'] = (df_clean['adr'] > CAPS['adr']).astype(int)
    df_clean['adr'] = df_clean['adr'].clip(upper=CAPS['adr'])

    # Lead time
    df_clean['lead_time_was_capped'] = (df_clean['lead_time'] > CAPS['lead_time']).astype(int)
    df_clean['lead_time'] = df_clean['lead_time'].clip(upper=CAPS['lead_time'])
    df_clean['lead_time_log'] = np.log1p(df_clean['lead_time'])

    # Stay nights
    df_clean['stays_weekend_was_capped'] = (df_clean['stays_in_weekend_nights'] > CAPS['stays_in_weekend_nights']).astype(int)
    df_clean['stays_in_weekend_nights'] = df_clean['stays_in_weekend_nights'].clip(upper=CAPS['stays_in_weekend_nights'])

    df_clean['stays_week_was_capped'] = (df_clean['stays_in_week_nights'] > CAPS['stays_in_week_nights']).astype(int)
    df_clean['stays_in_week_nights'] = df_clean['stays_in_week_nights'].clip(upper=CAPS['stays_in_week_nights'])

    # Guests
    df_clean['adults_was_capped'] = (df_clean['adults'] > LIMITS['adults']).astype(int)
    df_clean['adults'] = df_clean['adults'].clip(upper=LIMITS['adults'])

    df_clean['has_children'] = (df_clean['children'] > 0).astype(int)
    df_clean['children_was_capped'] = (df_clean['children'] > LIMITS['children']).astype(int)
    df_clean['children'] = df_clean['children'].clip(upper=LIMITS['children'])

    df_clean['has_babies'] = (df_clean['babies'] > 0).astype(int)
    df_clean['babies_was_capped'] = (df_clean['babies'] > LIMITS['babies']).astype(int)
    df_clean['babies'] = df_clean['babies'].clip(upper=LIMITS['babies'])

    # Booking changes
    df_clean['has_booking_changes'] = (df_clean['booking_changes'] > 0).astype(int)
    df_clean['booking_changes_was_capped'] = (df_clean['booking_changes'] > CAPS['booking_changes']).astype(int)
    df_clean['booking_changes'] = df_clean['booking_changes'].clip(upper=CAPS['booking_changes'])

    # Waiting list
    df_clean['has_waiting_list'] = (df_clean['days_in_waiting_list'] > 0).astype(int)
    df_clean['days_waiting_was_capped'] = (df_clean['days_in_waiting_list'] > CAPS['days_in_waiting_list']).astype(int)
    df_clean['days_in_waiting_list'] = df_clean['days_in_waiting_list'].clip(upper=CAPS['days_in_waiting_list'])
    df_clean['days_in_waiting_list_log'] = np.log1p(df_clean['days_in_waiting_list'])

    # Previous history (keep raw for traceability)
    df_clean['previous_cancellations_raw'] = df_clean['previous_cancellations']
    df_clean['previous_bookings_not_canceled_raw'] = df_clean['previous_bookings_not_canceled']

    df_clean['previous_cancellations_was_capped'] = (
        df_clean['previous_cancellations_raw'] > CAPS['previous_cancellations']
    ).astype(int)
    df_clean['previous_cancellations'] = df_clean['previous_cancellations_raw'].clip(upper=CAPS['previous_cancellations'])
    df_clean['has_prev_cancellations'] = (df_clean['previous_cancellations'] > 0).astype(int)
    df_clean['previous_cancellations_log'] = np.log1p(df_clean['previous_cancellations'])

    df_clean['previous_bookings_not_canceled_was_capped'] = (
        df_clean['previous_bookings_not_canceled_raw'] > CAPS['previous_bookings_not_canceled']
    ).astype(int)
    df_clean['previous_bookings_not_canceled'] = df_clean['previous_bookings_not_canceled_raw'].clip(upper=CAPS['previous_bookings_not_canceled'])
    df_clean['has_prev_not_canceled'] = (df_clean['previous_bookings_not_canceled'] > 0).astype(int)
    df_clean['previous_bookings_not_canceled_log'] = np.log1p(df_clean['previous_bookings_not_canceled'])

    df_clean['has_any_history'] = (
        (df_clean['previous_cancellations'] + df_clean['previous_bookings_not_canceled']) > 0
    ).astype(int)
    prev_total = df_clean['previous_cancellations'] + df_clean['previous_bookings_not_canceled']
    df_clean['prev_cancel_ratio'] = np.where(prev_total > 0, df_clean['previous_cancellations'] / prev_total, 0.0)

    # ---- Feature engineering used in EDA ----
    df_clean['arrival_month_num'] = df_clean['arrival_date_month'].map(MONTH_TO_NUM)

    df_clean['total_nights'] = df_clean['stays_in_weekend_nights'] + df_clean['stays_in_week_nights']
    df_clean['total_guests'] = df_clean['adults'] + df_clean['children'] + df_clean['babies']

    df_clean['is_room_changed'] = (
        df_clean['reserved_room_type'] != df_clean['assigned_room_type']
    ).astype(int)

    df_clean['arrival_date'] = pd.to_datetime(
        df_clean['arrival_date_year'].astype(str) + '-' +
        df_clean['arrival_month_num'].astype('Int64').astype(str).str.zfill(2) + '-' +
        df_clean['arrival_date_day_of_month'].astype(str).str.zfill(2),
        errors='coerce'
    )

    df_clean['revenue_at_risk'] = (df_clean['adr'] * df_clean['total_nights']).round(2)

    if country_lookup is not None:
        lookup = country_lookup.rename(columns={'country_code': 'country'})
        df_clean = df_clean.merge(lookup, on='country', how='left')
        if 'continent' in df_clean.columns:
            df_clean['continent'] = df_clean['continent'].fillna('Other')
        if 'region' in df_clean.columns:
            df_clean['region'] = df_clean['region'].fillna('Other')

    df_clean['lead_time_bucket'] = pd.cut(
        df_clean['lead_time'],
        bins=[-1, 7, 30, 90, 180, 365, 9999],
        labels=['Same week', '1-4 weeks', '1-3 months', '3-6 months', '6-12 months', '12+ months']
    )

    # Same EDA behavior: deduplicate at end
    if drop_duplicates:
        df_clean = df_clean.drop_duplicates().reset_index(drop=True)

    return df_clean


In [ ]:
# Example usage
raw_path = './data/hotel_bookings.csv'
lookup_path = './data/country_lookup.csv'

raw_df = pd.read_csv(raw_path)
lookup_df = pd.read_csv(lookup_path)

processed_df = preprocess_new_data(raw_df, country_lookup=lookup_df, drop_duplicates=True)

print(f'Processed shape: {processed_df.shape}')
print('Sample engineered columns:')
print([c for c in [
    'adr_was_capped','lead_time_log','has_waiting_list','days_in_waiting_list_log',
    'has_prev_cancellations','has_prev_not_canceled','has_any_history','prev_cancel_ratio',
    'arrival_month_num','total_nights','total_guests','is_room_changed','lead_time_bucket'
] if c in processed_df.columns])


In [ ]:
# Save engineered output for modeling/inference
processed_df.to_csv('./data/hotel_bookings_feature_engineered.csv', index=False)
print('Saved: ./data/hotel_bookings_feature_engineered.csv')
